# 07 - Parameter Batch Evaluation (Outer Product)

This notebook computes the expectation value for the **Cartesian product (outer product)** of a parameter batch and an embedding batch, using the `Circuit` builder API.

Given:
- `thetas` of shape `(p_bs, N_params)`
- `embedding` of shape `(e_bs, N_emb)`

`qc.expvals(thetas, embedding=embedding)` returns shape `(e_bs, p_bs, N_obs)`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

from padopauli import Circuit

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float64
print(f"Using device: {device}")

Using device: cuda


In [2]:
# 1. Setup a simple circuit with both embedding and parameters
n_qubits = 2
qc = Circuit(n_qubits=n_qubits)
qc.rx(0, embedding_idx=0)
qc.ry(1, param_idx=0)

observables = [("Z", [0]), ("Z", [1])]
n_obs = len(observables)

qc.compile(observables=observables, preset='hybrid' if device == 'cuda' else 'cpu')
qc.draw()

propagate:   0%|          | 0/2 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 4


zero-filter:   0%|          | 0/2 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 2 (50.000000% of peak)
q0:─RXx₀─<Z>────
q1:─RYθ₀─────<Z>

compiled: preset=hybrid, max_weight=default, zero_filter=True, n_params=1, n_embedding=1


In [3]:
# 2. Define batches
e_bs = 3
p_bs = 3

embedding_batch = torch.tensor([[0], [torch.pi/2], [torch.pi]], dtype=dtype, device=device)
thetas_batch = torch.tensor([[0], [torch.pi/2], [torch.pi]], dtype=dtype, device=device)


print(f"Embedding batch shape: {embedding_batch.shape}")
print(f"Thetas batch shape: {thetas_batch.shape}")

# 3. Compute batched expvals (outer product over embedding x parameters).
res_batched = qc.expvals(thetas_batch, embedding=embedding_batch)
print(f"Result shape: {res_batched.shape}")

assert res_batched.shape == (e_bs, p_bs, n_obs), f"Shape mismatch! Expected {(e_bs, p_bs, n_obs)}, got {res_batched.shape}"


Embedding batch shape: torch.Size([3, 1])
Thetas batch shape: torch.Size([3, 1])


Result shape: torch.Size([3, 3, 2])


In [4]:
# 4. Verify results with nested loops (reference)
res_ref = torch.zeros(e_bs, p_bs, n_obs, dtype=dtype, device=device)

for i in range(e_bs):
    for j in range(p_bs):
        # Evaluate single combination
        single_val = qc.expvals(thetas_batch[j], embedding=embedding_batch[i])
        res_ref[i, j] = single_val

diff = torch.abs(res_batched - res_ref)
max_diff = torch.max(diff).item()
print(f"Max absolute difference: {max_diff:.2e}")

is_correct = torch.allclose(res_batched, res_ref.to(res_batched.dtype), atol=1e-12)
print(f"Numerical verification passed: {is_correct}")
assert is_correct

Max absolute difference: 0.00e+00
Numerical verification passed: True


In [5]:
# 5. Check compatibility: Only one batch
res_only_p = qc.expvals(thetas_batch, embedding=embedding_batch[0])
print(f"Only thetas batch shape: {res_only_p.shape}")
assert res_only_p.shape == (p_bs, n_obs)

res_only_e = qc.expvals(thetas_batch[0], embedding=embedding_batch)
print(f"Only embedding batch shape: {res_only_e.shape}")
assert res_only_e.shape == (e_bs, n_obs)

res_none = qc.expvals(thetas_batch[0], embedding=embedding_batch[0])
print(f"No batch shape: {res_none.shape}")
assert res_none.shape == (n_obs,)

res_all = qc.expvals(thetas_batch, embedding=embedding_batch)
print(f"All batch shape: {res_all.shape}")
assert res_all.shape == (e_bs, p_bs, n_obs)

Only thetas batch shape: torch.Size([3, 2])
Only embedding batch shape: torch.Size([3, 2])
No batch shape: torch.Size([2])
All batch shape: torch.Size([3, 3, 2])
